# درب السيرة — generate the narration audio (run once)

Settings (right sidebar): **Accelerator: GPU T4 x2**, **Internet: On**.
If your GitHub repo is private: Add-ons → Secrets → add `GITHUB_TOKEN`.

1. Edit `REPO` in the next cell.
2. Run the cells **up to "Test one chapter"** interactively and listen.
3. Then **Save Version → Save & Run All** so the full run finishes in the background.
4. When it's done: open the notebook's **Output** tab → download `audio.zip` → unzip it into the project's `data/` folder.


In [ ]:
REPO = "https://github.com/shazaosama785/darb-alseerah.git"   # <- your repo
BRANCH = "main"
TEST_CHAPTER = "ch02"      # a chapter with Quran verses, for the quick check
FULL_RUN = True            # False = only the quick test


In [ ]:
import torch, subprocess, os
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2"
GPUS = torch.cuda.device_count()
print("GPUs:", GPUS, torch.cuda.get_device_name(0))


In [ ]:
# Get the project (uses the GITHUB_TOKEN secret if the repo is private)
url = REPO
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO.replace("https://", f"https://{token}@")
except Exception:
    pass
os.chdir("/kaggle/working")
subprocess.run("rm -rf project", shell=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "project"], check=True)
os.chdir("/kaggle/working/project/tts")
print(os.listdir("."))


In [ ]:
# Install the model packages. pregenerate.py runs in a NEW process, so no kernel restart is needed.
subprocess.run('pip install -q coqui-tts "transformers==4.57.6"', shell=True)


## Test one chapter
Listen to one event with a Quran verse before spending hours on the full book.

In [ ]:
r = subprocess.run(["python", "scripts/pregenerate.py", "--data", "../data", "--chapters", TEST_CHAPTER,
                    "--out", "/kaggle/working/test_audio"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:])


In [ ]:
import json, glob
from IPython.display import Audio, display
for tl in sorted(glob.glob("/kaggle/working/test_audio/*.json"))[:3]:
    items = json.load(open(tl, encoding="utf-8"))["items"]
    print(tl, "| verses:", sum(i["kind"] == "quran" for i in items))
    for i in items[:4]:
        print("  ", i["kind"], i.get("start", i.get("at")), i["text"][:70])
    display(Audio(tl.replace(".json", ".mp3")))


## Full book
One process per GPU, each takes half the events. Resumable: finished events are skipped.

In [ ]:
if FULL_RUN:
    jobs = [subprocess.Popen(
                ["python", "scripts/pregenerate.py", "--data", "../data", "--shard", f"{g}/{GPUS}"],
                env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)},
                stdout=open(f"/kaggle/working/gen{g}.log", "w"), stderr=subprocess.STDOUT)
            for g in range(GPUS)]
    codes = [j.wait() for j in jobs]
    print("exit codes:", codes)
    for g in range(GPUS):
        print(open(f"/kaggle/working/gen{g}.log").read()[-600:])


In [ ]:
if FULL_RUN:
    subprocess.run("cd ../data && zip -qr /kaggle/working/audio.zip audio", shell=True, check=True)
    n = len(glob.glob("../data/audio/*.mp3"))
    print(f"{n} narrations ->", round(os.path.getsize('/kaggle/working/audio.zip') / 1e6), "MB  (download audio.zip from the Output tab)")
